# Day 1 · Section 8: Causal Attention

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 8.1–8.4

Make a causal mask, apply it before softmax, distinguish a future-token mask from a padding mask, and verify that changing a future token does not alter earlier outputs. Both NumPy and PyTorch paths are shown.


In [ ]:
import sys, subprocess, numpy as np
try:
    import torch
    DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',DEVICE)
except ImportError:
    torch=None;DEVICE=None
    print('PyTorch unavailable locally; NumPy path remains runnable. Colab normally includes PyTorch.')
rng=np.random.default_rng(7)


### 8.1–8.2 · Future tokens must be invisible

For a sequence of length L, row i may see columns 0 through i. The upper triangle marks disallowed future positions. Mask scores *before* softmax; do not set probabilities to zero after softmax without renormalizing.


In [ ]:
length=4
future_mask=np.triu(np.ones((length,length),dtype=bool),k=1)
print('1 means visible:\n',(~future_mask).astype(int))
assert np.array_equal((~future_mask).astype(int),np.tril(np.ones((4,4),dtype=int)))
X=np.array([[1.,0.],[0.,1.],[1.,1.],[.5,.5]])
scores=X@X.T/np.sqrt(X.shape[-1])
masked=np.where(future_mask,-np.inf,scores)
def softmax_np(z):
    z=z-z.max(axis=-1,keepdims=True);e=np.exp(z);return e/e.sum(axis=-1,keepdims=True)
weights=softmax_np(masked);output=weights@X
print('causal weights:\n',np.round(weights,3))
assert np.all(weights[future_mask]==0) and np.allclose(weights.sum(axis=1),1)


### NumPy versus PyTorch masked attention

PyTorch's `masked_fill(mask, -inf)` hides positions where `mask` is true. Keep mask shape `[sequence, sequence]` for this single example; batched attention may need broadcast dimensions.


In [ ]:
if torch is not None:
    Xt=torch.tensor(X,dtype=torch.float32,device=DEVICE)
    score_t=Xt@Xt.transpose(-2,-1)/np.sqrt(Xt.shape[-1])
    future_t=torch.triu(torch.ones((length,length),dtype=torch.bool,device=DEVICE),diagonal=1)
    weight_t=torch.softmax(score_t.masked_fill(future_t,float('-inf')),dim=-1)
    result_t=weight_t@Xt
    print('torch weights:\n',weight_t.cpu().numpy().round(3))
    assert np.allclose(result_t.cpu().numpy(),output,atol=1e-6)
else:print('PyTorch masked-attention cell skipped.')


### Test left-to-right independence

Replace the final token's features with a very different vector. Outputs at earlier positions should remain unchanged under a causal mask. They *do* change when all positions may attend to all positions.


In [ ]:
def attention_output(x,causal=True):
    score=x@x.T/np.sqrt(x.shape[-1])
    if causal:score=np.where(np.triu(np.ones(score.shape,dtype=bool),k=1),-np.inf,score)
    return softmax_np(score)@x
changed=X.copy();changed[-1]=np.array([100.,-100.])
causal_old=attention_output(X);causal_new=attention_output(changed)
unmasked_old=attention_output(X,False);unmasked_new=attention_output(changed,False)
print('early causal output unchanged:',np.allclose(causal_old[:-1],causal_new[:-1]))
print('early unmasked output unchanged:',np.allclose(unmasked_old[:-1],unmasked_new[:-1]))
assert np.allclose(causal_old[:-1],causal_new[:-1])


### 8.3 · Padding hides nonexistent Keys

A padding mask hides positions added merely to make batch lengths equal. Combine it with a causal mask. The padded *Query* row is not a meaningful prediction and must also be ignored when computing a loss.


In [ ]:
valid_keys=np.array([True,True,True,False])
combined_mask=future_mask | (~valid_keys[None,:])
combined_weights=softmax_np(np.where(combined_mask,-np.inf,scores))
print('padding key column:',combined_weights[:,3])
assert np.all(combined_weights[:,3]==0)
print('padded query row is still computed:',combined_weights[3],'; exclude it downstream')
if torch is not None:
    pad_t=torch.tensor(~valid_keys,device=DEVICE)
    combined_t=future_t|pad_t[None,:]
    weights_t=torch.softmax(score_t.masked_fill(combined_t,float('-inf')),dim=-1)
    assert np.allclose(weights_t.cpu().numpy(),combined_weights,atol=1e-6)


### Fully masked rows need an explicit policy

If every Key is masked for one Query, softmax of all `−inf` is undefined and may produce NaNs. In a real implementation, avoid such rows or handle them before softmax. This can happen with certain padding layouts or custom masks.


In [ ]:
all_hidden=np.array([[-np.inf,-np.inf]])
print('all Keys hidden:',np.isneginf(all_hidden).all())
print('We do not run softmax on this row; it has no valid probability distribution.')
# Exercise: create a mask whose last query row has no visible keys, then handle it explicitly.


## Checks

1. Why does the causal mask use the upper triangle for disallowed positions?
2. Why mask before softmax? What is the difference between a hidden Key and an ignored padded Query?
3. Change the second token. Which output rows may change under causal attention?
